# Track 10 — Capstone: Data Analyst (CSV 질의 · 샌드박스 vs 허용목록)

## 자유 SQL의 위험과 두 가지 방어

데이터 분석 에이전트에 자유 SQL을 그대로 맡기면 원본 데이터 유출(`SELECT *`)이나 파괴(`DROP`)가 발생할 수 있습니다. 이 캡스톤은 두 방어 전략의 차이를 비교합니다.

- **샌드박스(격리):** 임의 SQL을 실행하되 읽기전용 환경에 가둡니다. 파괴는 막지만 전수 덤프는 막지 못합니다.
- **허용목록(최소권한):** 임의 SQL을 제공하지 않고 검증된 집계 op만 노출합니다. 유출과 파괴를 모두 줄입니다.

목표는 단순합니다. **집계만 노출하고 원본 행은 노출하지 않습니다.** 그래서 최종 도구는 허용목록 기반 `analyze_csv`로 구성합니다.

## 이 노트북에서 보여줄 것

| Session | 보여주는 것 | 목적 |
|---|---|---|
| 1. Setup | facade 시작 · 골든 로더 · 회귀 헬퍼 · 패키지 저장 헬퍼 | 공통 준비 |
| 2. 위험 → 방어 | `free_sql` 위험 → 읽기전용 샌드박스 → 허용목록 `analyze_csv` | 격리와 최소권한 비교 |
| 3. 패키지 | 집계 결과와 정적 메트릭 저장 | 제출·회귀 산출물 마감 |

## 이 노트북을 마치면

- 자유 SQL의 유출·파괴 위험을 설명할 수 있습니다.
- 샌드박스와 허용목록의 차이를 실제 출력으로 비교할 수 있습니다.
- 허용된 op만 실행하는 `analyze_csv` 도구를 등록하고 스키마 enum·도구 가드 2층 방어를 확인할 수 있습니다.

**산출물:** `_out/04/capstone_package.json`  
**실행 조건:** 라이브 LLM 호출이 없어 API 키 없이 전 구간이 실행됩니다.

> 요약: 자유 SQL 위험을 보여준 뒤 허용목록 집계 도구로 바꾸고, 결과를 회귀 패키지로 마감하는 캡스톤입니다.


## Session 1. Setup


### Session 1-1. Setup

**하는 일:** 경로·클라이언트·공통 헬퍼를 준비합니다.

**정상:** `exaone 0.1.0 | HAS_API True` 같은 한 줄이 출력됩니다(키가 없으면 `HAS_API False`). 헬퍼 정의만 실행되어 그 밖의 출력은 없습니다.

**의미:** 이후 단계에서 쓸 경로·클라이언트가 맞는지 먼저 봅니다.

In [ ]:
import json
import os
from datetime import datetime, timezone
from pathlib import Path

import logging

# (en) Quiet library logs so the notebook output stays readable.
# (kr) 라이브러리 로그를 줄여 노트북 출력을 읽기 쉽게 한다.
for _log_name in ("exaone", "exaone.llm", "exaone.llm.exaone_client", "urllib3"):
    logging.getLogger(_log_name).setLevel(logging.ERROR)

# (en) Facade-only startup; requires editable install at the repo root.
# (kr) `exaone` facade로 시작한다. 저장소 루트에서 editable 설치가 필요하다.
try:
    import exaone
except ModuleNotFoundError as exc:
    raise ModuleNotFoundError(
        "`exaone`이 설치되지 않았습니다. 저장소 루트에서 "
        "pip install -r requirements.txt && pip install -e ./exaone 후 커널을 재시작하세요."
    ) from exc

exaone.load_project_env()
ROOT = exaone.project_root()
TRACK10 = ROOT / "recipes" / "track10_ax_capstones"
DATA = TRACK10 / "data"
API_KEY = os.environ.get("EXAONE_API_KEY", "").strip()
BASE_URL = os.environ.get("EXAONE_BASE_URL", "").strip() or "http://localhost:8000/v1"
MODEL = os.environ.get("EXAONE_MODEL", "").strip() or exaone.llm.ExaoneClient.DEFAULT_MODEL
HAS_API = bool(API_KEY)
client = None
if HAS_API:
    client = exaone.llm.ExaoneAPIClient(base_url=BASE_URL, model=MODEL, api_key=API_KEY)
print("exaone", exaone.__version__, "| HAS_API", HAS_API)

def load_capstone_golden(tag: str) -> list[dict]:
    # (en) Load golden rows for this capstone id or shared "all" rows.
    # (kr) 해당 캡스톤과 공통 "all" 골든 사례를 함께 불러온다.
    rows: list[dict] = []
    for line in (DATA / "capstone_golden.jsonl").read_text(encoding="utf-8").splitlines():
        if not line.strip():
            continue
        row = json.loads(line)
        if row.get("capstone") in (tag, "all"):
            rows.append(row)
    return rows


def regression_m1_m6_m9(rows: list[dict]) -> dict:
    # (en) Score golden rows on M1 (exact), M6 (loose schema), M9 (stub length-ratio judge).
    # (kr) 골든 행을 M1(정확 일치)·M6(느슨 스키마)·M9(스텁 길이비 심판)로 채점한다.
    from eval.metrics.m1_task_success import TaskGold
    from eval.metrics.m6_schema_adherence import SchemaSpec
    from eval.metrics.m9_faithfulness import LengthRatioJudge
    from eval.metrics import m1_task_success, m6_schema_adherence
    from eval.metrics.types import TrialResult

    m1s, m6s, m9s = [], [], []
    cases = []
    for row in rows:
        tid = row["id"]
        content = row.get("trial_content") or str(row.get("expected_answer", ""))
        tr = TrialResult(
            trial_id=f"cap-{tid}",
            task_id=tid,
            dataset="track10.golden",
            runner="capstone",
            final_content=content,
        )
        m1 = m6 = m9 = None
        if row.get("expected_answer") is not None:
            m1 = m1_task_success.score_trial_exact(tr, TaskGold(task_id=tid, answer=row["expected_answer"]))
            m1s.append(m1)
        rk = row.get("required_keys")
        if rk:
            _, loose = m6_schema_adherence.score_trial(tr, SchemaSpec(required_keys=rk))
            m6 = loose
            m6s.append(1.0 if loose else 0.0)
        if row.get("grounding_context"):
            m9 = LengthRatioJudge()(trial=tr, gold={"context": row["grounding_context"]})
            m9s.append(m9)
        cases.append({"id": tid, "M1": m1, "M6": m6, "M9": m9})
    mean = lambda xs: sum(xs) / len(xs) if xs else 0.0
    return {"n": len(rows), "M1_mean": mean(m1s), "M6_loose_mean": mean(m6s), "M9_mean": mean(m9s), "cases": cases}


def save_package(capstone_nb: str, body: dict) -> Path:
    # (en) Write capstone_package.json under <track>/_out/<nb>/ (absolute path, CWD-independent).
    # (kr) 절대경로로 <track>/_out/<nb>/capstone_package.json을 저장한다(커널 CWD와 무관).
    out_dir = TRACK10 / "_out" / capstone_nb
    out_dir.mkdir(parents=True, exist_ok=True)
    slo = exaone.observability.SLOSpec(
        name=f"capstone-{capstone_nb}",
        p95_chat_latency_ms=8000,
        structured_output_success_min="95%",
        notes="Track 10 capstone — adjust per deployment.",
    )
    payload = {
        "generated_at": datetime.now(timezone.utc).isoformat(timespec="seconds"),
        "capstone_id": capstone_nb,
        "slo": slo.to_dict(),
        **body,
    }
    path = out_dir / "capstone_package.json"
    path.write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding="utf-8")
    print("saved", path.resolve())
    return path


**출력 해석:** `exaone <버전> | HAS_API True/False` 한 줄이 보이면 준비 완료입니다.

- `TRACK10`·`DATA`가 **절대경로**로 잡혀, 이후 셀이 어느 CWD에서 실행돼도 같은 파일을 읽고 같은 위치(`_out/04/`)에 저장합니다.
- 이 노트북은 라이브 LLM 호출이 없어 `HAS_API`가 `False` 여도 전 구간이 동작합니다 — 플래그는 환경 표시용입니다.


## Session 2. 자유 SQL의 위험 → 두 가지 방어 (샌드박스 · 허용목록)

### Session 2-1. 방어 없음 — 자유 SQL의 위험

**하는 일:** 비교 기준으로, 임의 SQL을 그대로 실행하는 `free_sql` 도구를 인메모리 SQLite(매출 테이블) 위에서 돌려 봅니다.

**정상:** 세 줄: `집계 요청: [(2750,)]`(정상 집계), `전수 덤프: [('서울','A',1200), …]`(원본 전체 행 노출), `DROP 이후: no such table: sales`(테이블 삭제 뒤 조회 실패).

**의미:** 아무 제약이 없으면 "합계"뿐 아니라 **전수 덤프(유출)** 와 **`DROP`(파괴)** 까지 무엇이든 실행됩니다 — 이어지는 두 셀에서 **샌드박스(격리)** 와 **허용목록(최소권한)** 두 방어책이 각각 무엇을 막는지 비교합니다.

In [ ]:
import csv
import sqlite3

# (en) Build one in-memory SQLite table from the CSV (a stand-in for a real analytics DB).
# (kr) CSV로 인메모리 SQLite 테이블을 만든다(실제 분석 DB 대체 환경).
danger_conn = sqlite3.connect(":memory:")
danger_conn.execute("CREATE TABLE sales(region TEXT, product TEXT, revenue INT)")
_rows = list(csv.DictReader((DATA / "sales_sample.csv").open(encoding="utf-8")))
danger_conn.executemany("INSERT INTO sales VALUES (?, ?, ?)", [(r["region"], r["product"], int(r["revenue"])) for r in _rows])


def free_sql(query: str) -> list:
    # (en) ANTI-PATTERN: this "free SQL" tool runs ANY statement the model emits — full power, no allow-list.
    # (kr) 안티패턴: 이 "자유 SQL" 도구는 모델이 보낸 문장을 그대로 실행한다. 허용목록이 없어 권한이 과도하다.
    return danger_conn.execute(query).fetchall()


# (en) A legitimate aggregate request looks perfectly fine.
# (kr) 정상적인 집계 요청은 멀쩡해 보인다.
print("집계 요청:", free_sql("SELECT SUM(revenue) FROM sales"))

# (en) But the SAME tool also dumps every raw row — full-table / sensitive-data exposure.
# (kr) 그러나 같은 도구가 원본 전체 행도 그대로 덤프한다. 전수·민감정보 노출이다.
print("전수 덤프:", free_sql("SELECT * FROM sales"))

# (en) And it will even destroy data: nothing stops a DROP TABLE.
# (kr) DROP TABLE도 막지 못해 데이터를 파괴할 수 있다.
free_sql("DROP TABLE sales")
try:
    free_sql("SELECT SUM(revenue) FROM sales")
except sqlite3.OperationalError as exc:
    print("DROP 이후:", exc)

**출력 해석:** `free_sql`은 "무엇이든 실행"하므로 한 도구로 정상·위험을 가리지 않습니다.

- `집계 요청: [(2750,)]` — 합계만 보면 멀쩡해 보입니다.
- `전수 덤프: [('서울','A',1200), …]` — 같은 도구가 **원본 전체 행**을 그대로 돌려줍니다(유출). 실제 데이터라면 고객·매출 원장 전체가 노출되는 셈입니다.
- `DROP 이후: no such table: sales` — `DROP TABLE` 한 줄로 테이블이 사라져 이후 조회가 실패합니다(파괴). 되돌릴 수 없는 작업이 아무 제지 없이 실행됐습니다.
- 이 위험을 줄이는 두 방어책을 다음에 비교합니다 — **① 샌드박스**: 임의 SQL을 격리(읽기전용) 실행해 파괴를 막음, **② 허용목록**: 임의 SQL을 아예 안 주고 검증된 op 만 노출.

### Session 2-2. 방어① 샌드박스 — 임의 SQL을 "격리 실행"

**하는 일:** 같은 임의 SQL을 이번엔 **읽기전용 연결**(`PRAGMA query_only=ON`)에서 실행합니다 — 쿼리는 그대로 실행되되 엔진이 쓰기·DDL을 거부합니다.

**정상:** 세 줄: `집계: [(2750,)]`(통과), `DROP 차단: attempt to write a readonly database`(파괴 거부), `전수 덤프(여전히 통과): [('서울','A',1200), …]`(읽기는 그대로).

**의미:** 샌드박스는 "실행하되 가둔다" — **파괴는 막지만 유출(전수 덤프)은 못 막습니다.** 격리만으로는 부족하다는 점이 핵심입니다.

In [ ]:
# (en) DEFENSE #1 — SANDBOX: run the SAME arbitrary SQL, but inside a read-only connection.
# (kr) 방어① 샌드박스: 같은 임의 SQL을 읽기전용 연결 안에서 실행한다.
sandbox_conn = sqlite3.connect(":memory:")
sandbox_conn.execute("CREATE TABLE sales(region TEXT, product TEXT, revenue INT)")
sandbox_conn.executemany("INSERT INTO sales VALUES (?, ?, ?)", [(r["region"], r["product"], int(r["revenue"])) for r in _rows])
# (en) From here the engine itself rejects any write or DDL (DROP/INSERT/UPDATE/CREATE).
# (kr) 이 시점부터 엔진이 쓰기·DDL(DROP/INSERT/UPDATE/CREATE)을 직접 거부한다.
sandbox_conn.execute("PRAGMA query_only = ON")


def sandboxed_sql(query: str) -> list:
    # (en) Arbitrary SQL still RUNS — the sandbox confines it, it does not vet the intent.
    # (kr) 임의 SQL은 여전히 실행된다 — 샌드박스는 가둘 뿐, 의도를 검증하지는 않는다.
    return sandbox_conn.execute(query).fetchall()


# (en) Aggregate runs fine.
# (kr) 집계는 정상 동작.
print("집계:", sandboxed_sql("SELECT SUM(revenue) FROM sales"))

# (en) Destruction is now blocked by the read-only engine — what the sandbox buys you.
# (kr) 파괴는 읽기전용 엔진이 막는다 — 샌드박스가 주는 이득.
try:
    sandboxed_sql("DROP TABLE sales")
except sqlite3.OperationalError as exc:
    print("DROP 차단:", exc)

# (en) BUT a full dump still passes: a read-only sandbox stops destruction, NOT exfiltration.
# (kr) 그러나 전수 덤프는 여전히 통과: 읽기전용 샌드박스는 파괴는 막아도 유출은 못 막는다.
print("전수 덤프(여전히 통과):", sandboxed_sql("SELECT * FROM sales"))

**출력 해석:** 샌드박스는 임의 SQL을 **실행하되 가둡니다** — 파괴는 막고, 유출은 못 막습니다.

- `집계: [(2750,)]` — 정상 쿼리는 그대로 동작합니다.
- `DROP 차단: attempt to write a readonly database` — `query_only=ON`이라 엔진이 쓰기·DDL을 거부합니다. `free_sql` 에서 통했던 `DROP`이 여기선 막힙니다 — 이것이 샌드박스(격리)가 주는 이득입니다.
- `전수 덤프(여전히 통과): […]` — 하지만 `SELECT *`는 읽기라 그대로 통과해 **원본 전체 행이 노출**됩니다. **읽기전용 샌드박스는 파괴는 막아도 유출은 못 막습니다.**
- 즉 샌드박스는 "임의 실행이 꼭 필요할 때" 피해를 줄이는 전략일 뿐, "무엇을 읽을 수 있는가"까지 통제하려면 추가 제약(열·행 제한)이 필요합니다. 다음 셀의 허용목록은 다른 전략으로 이 한계를 넘습니다.

### Session 2-3. 방어② 허용목록(최소권한) — `analyze_csv`

**하는 일:** 전략을 바꿔, **임의 SQL을 아예 제공하지 않고** 검증된 집계 op(`ALLOWED_OPS`)만 노출하는 `analyze_csv`를 등록합니다. 자연어 질의를 op 로 매핑해 호출하고, 목록 밖 op 는 두 층(스키마 enum · 도구 가드)에서 막아 봅니다.

**정상:** 질의 3건 → 답(`count_rows→4`·`sum_revenue→2750`·`top_region→서울:1650`), 이어 목록 밖 op(`drop_table`) 두 줄 거부 — `거부(스키마 enum): … not one of […]`와 `거부(도구 가드): outcome=validation_error error=op not allowed`.

**의미:** 샌드박스와 달리, **위험한 의도(전수 덤프·`DROP`)는 op 로 표현조차 불가능**합니다 — 집계 3종 외엔 호출할 길이 없으니 유출·파괴가 **둘 다** 차단됩니다. 이 캡스톤의 목표(집계만 노출)엔 허용목록(최소권한)이 정답입니다.

In [ ]:

import csv

CSV_PATH = DATA / "sales_sample.csv"
# (en) Ordered allow-list: a tuple keeps op order deterministic across runs (a set would not).
# (kr) 순서 보장 허용 목록: 튜플이라 op 순서가 실행마다 결정적이다(set 은 그렇지 않다).
ALLOWED_OPS = ("count_rows", "sum_revenue", "top_region")


def safe_analyze(_n: str, args: dict) -> dict:
    # (en) Sandboxed CSV aggregation: only allow-listed ops run; anything else is a validation error.
    # (kr) 샌드박스 CSV 집계: 허용 목록의 op 만 실행하고, 그 밖은 검증 오류로 막는다.
    op = args.get("op")
    if op not in ALLOWED_OPS:
        return exaone.tools.ToolResult.validation_error(source="analyze_csv", error="op not allowed").to_dict()
    rows = list(csv.DictReader(CSV_PATH.open(encoding="utf-8")))
    if op == "count_rows":
        return exaone.tools.ToolResult.success(content=str(len(rows)), source="analyze_csv").to_dict()
    if op == "sum_revenue":
        total = sum(int(r["revenue"]) for r in rows)
        return exaone.tools.ToolResult.success(content=str(total), source="analyze_csv").to_dict()
    by_region: dict[str, int] = {}
    for r in rows:
        by_region[r["region"]] = by_region.get(r["region"], 0) + int(r["revenue"])
    top = max(by_region, key=by_region.get)
    return exaone.tools.ToolResult.success(content=f"{top}:{by_region[top]}", source="analyze_csv").to_dict()


ANALYZE_SCHEMA = {
    "type": "function",
    "function": {
        "name": "analyze_csv",
        "description": "Run allow-listed aggregation on sales_sample.csv",
        "parameters": {
            "type": "object",
            "required": ["op"],
            "properties": {"op": {"type": "string", "enum": list(ALLOWED_OPS)}},
            "additionalProperties": False,
        },
    },
}
reg = exaone.tools.ToolRegistry()
reg.register(exaone.tools.tool_from_callable("analyze_csv", ANALYZE_SCHEMA, safe_analyze))
# (en) Run every allow-listed op once; this dict feeds the capstone package in Session 3.
# (kr) 허용된 op를 한 번씩 실행한다; 이 딕셔너리가 Session 3의 캡스톤 패키지에 들어간다.
results = {op: reg.execute("analyze_csv", {"op": op}) for op in ALLOWED_OPS}

# (en) Learner view: map a natural-language question to its op and surface the answer (offline, no LLM).
# (kr) 학습자용 뷰: 자연어 질문을 op 로 매핑해 답을 보여준다(오프라인, LLM 불요).
QUERIES = {"행은 몇 개인가요?": "count_rows", "총 매출은 얼마인가요?": "sum_revenue", "매출 1위 지역은?": "top_region"}
for q, op in QUERIES.items():
    print(f"질의: {q} → op={op} → 답: {results[op]['content']}")

# (en) Sandbox block, layer 1 — the registry validates args against the schema enum BEFORE the tool runs.
# (kr) 샌드박스 차단 1층 — 레지스트리가 도구 실행 전에 인자를 스키마 enum 으로 먼저 검증한다.
denied_schema = reg.execute("analyze_csv", {"op": "drop_table"})
print(f"거부(스키마 enum): {denied_schema.get('error')}")

# (en) Sandbox block, layer 2 — called directly, the tool's own allow-list guard returns a validation error.
# (kr) 샌드박스 차단 2층 — 도구를 직접 불러도 자체 허용목록 가드가 검증 오류를 돌려준다.
denied_guard = safe_analyze("analyze_csv", {"op": "drop_table"})
print(f"거부(도구 가드): outcome={denied_guard['outcome']} error={denied_guard['error']}")


**출력 해석:** 자연어 질의 3건이 허용 op 로 매핑돼 답을 돌려주고, 목록 밖 op 는 **두 층**에서 차단됩니다.

- `행은 몇 개인가요? → count_rows → 4`, `총 매출은 얼마인가요? → sum_revenue → 2750`(1200+800+450+300), `매출 1위 지역은? → top_region → 서울:1650`(서울 1200+450) — 답은 입력 CSV에만 의존하므로 값은 실행마다 동일합니다.
- **세 전략 비교:** `free_sql`은 덤프·DROP 모두 통과(위험), 읽기전용 샌드박스는 DROP은 막아도 **덤프는 통과**, 허용목록은 덤프·DROP 에 해당하는 op 가 **아예 없어 둘 다 불가능**합니다 — 목표가 "집계만 노출"이면 최소권한 허용목록이 가장 강한 보장입니다.
- `거부(스키마 enum)` — `reg.execute`가 도구 실행 **전에** `op` enum 으로 검증 → `drop_table`은 `_exaone_tool_failure` 로 막혀 본문에 도달조차 못 함(**1차 방어선**).
- `거부(도구 가드)` — 레지스트리 없이 직접 불러도 `safe_analyze`의 `if op not in ALLOWED_OPS`가드가 `outcome=validation_error` 반환(**2차 방어선**, defense-in-depth).
- 세 op 의 `ToolResult`(`results`)는 Session 3의 캡스톤 패키지(`csv_results`)로 전달됩니다.

## Session 3. 패키지


### Session 3-1. 패키지

**하는 일:** 회귀 점수를 출력하고, 회귀·제출용 패키지 JSON을 저장합니다.

**정상:** `regression n=22 | M1=0.57 M6=0.50 M9(stub)=0.53`과 `saved …/_out/04/capstone_package.json` 두 줄이 출력됩니다.

**의미:** 캡스톤 제출·회귀용 결과를 화면으로 확인하고 한 파일로 묶습니다.

In [ ]:

regression = regression_m1_m6_m9(load_capstone_golden("04"))
# (en) Surface the static metric-demo numbers inline (metric MECHANICS, not the agent's score).
# (kr) 정적 메트릭 데모 수치를 화면에 보여준다(에이전트 성능이 아니라 메트릭 동작 예시).
print(f"regression n={regression['n']} | M1={regression['M1_mean']:.2f} M6={regression['M6_loose_mean']:.2f} M9(stub)={regression['M9_mean']:.2f}")
pkg_path = save_package("04", {"csv_results": results, "regression": regression, "session_trace": [{"event": "analyze", "ops": list(ALLOWED_OPS)}]})


**출력 해석:** `regression n=22 | M1=0.57 M6=0.50 M9(stub)=0.53`과 `saved … _out/04/capstone_package.json` 두 줄이 출력되면 이 캡스톤 패키지가 완성된 것입니다.

- `regression` 수치는 공통 골든 `all` 22행을 채점하는 **메트릭 동작 데모**일 뿐 데이터 분석 도구의 성능이 아닙니다(`04` 전용 행은 아직 없음): `M1=0.57` = `expected_answer`가 있는 7행 중 **4건 정확 일치**, `M6=0.50` = `required_keys`가 있는 6행 중 **3건이 키 충족**, `M9(stub)=0.53` = `grounding_context`가 있는 5행에 대한 `LengthRatioJudge` **길이비 근사**(충실도 측정이 아닌 테스트 전용 스텁).
- 저장된 JSON 에는 `csv_results`(세 op 결과)·`regression`·`session_trace`·`slo`가 한데 묶여, 제출·회귀에 바로 쓸 수 있습니다.
- 경로는 `TRACK10/_out/04/` 절대경로라 커널의 CWD와 무관하게 항상 같은 위치에 저장됩니다.

## 마무리

이 캡스톤에서는 자유 SQL의 위험을 먼저 확인하고, 읽기전용 샌드박스와 허용목록 전략을 비교한 뒤 `analyze_csv` 결과를 `_out/04/capstone_package.json`으로 저장했습니다.

**핵심 정리**
- **방어 없음:** `free_sql`은 합계뿐 아니라 전수 덤프와 `DROP`까지 실행합니다.
- **샌드박스:** 읽기전용 연결은 `DROP`을 막지만 `SELECT *`는 그대로 통과합니다.
- **허용목록:** `count_rows`·`sum_revenue`·`top_region`만 노출하므로 위험한 의도를 op로 표현할 수 없습니다.
- **2층 차단:** 스키마 enum이 먼저 막고, 도구 내부 가드가 한 번 더 막습니다.
- **패키지:** 집계 결과, 정적 메트릭, trace, SLO를 한 파일로 묶습니다.

**한계**
- 읽기전용 샌드박스는 유출을 막지 못합니다. 열·행 제한이나 뷰 설계가 별도로 필요합니다.
- 자연어 질의는 고정 매핑입니다. 새 분석을 지원하려면 op와 스키마를 추가해야 합니다.
- 입력 CSV는 검증용 샘플이며 실제 규모·분포를 대표하지 않습니다.
- M9(`LengthRatioJudge`)는 테스트 전용 스텁입니다.

**다음:** 운영화는 `10_07` 프로덕션 하네스를 참고하세요. 다른 시나리오는 `10_05` 고객지원 라우터도 이어서 볼 수 있습니다.

## 체크포인트

- [ ] Session 2-1 자유 SQL의 전수 덤프·`DROP` 확인
- [ ] Session 2-2 샌드박스가 `DROP`은 막고 전수 덤프는 통과시키는지 확인
- [ ] Session 2-3 허용목록 op 3건 + 목록 밖 op 2층 차단 확인
- [ ] Session 3 회귀 요약 출력 + `_out/04/capstone_package.json` 저장
